# CSIRO Image2Biomass: INFERENCE Notebook (Winning Strategy)
- **Ensemble**: Auto-detects mix of DINO and ConvNeXt weights.
- **TTA**: Flip Augmentation enabled.


In [ ]:

import os
import sys
import glob
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from tqdm import tqdm

def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# --- WINNING STRATEGY CONFIGURATION ---
CONFIG = {
    'seed': 42,
    'img_size': 384,
    
    # ENSEMBLE STRATEGY: Train multiple backbones to average later
    # 1. 'vit_base_patch14_dinov2.lvd142m' (DINOv2 - Shape/Context)
    # 2. 'convnextv2_base.fcmae'           (ConvNeXt - Texture)
    'backbone': 'vit_base_patch14_dinov2.lvd142m', 
    
    'batch_size': 8,
    'epochs': 15, # Increased for Mixup
    'lr': 1e-4,
    'num_workers': 0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'n_folds': 5,
    'target_cols': ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g'],
    'train_csv': 'train.csv',
    'test_csv': 'test.csv',
    'img_dir': 'images/',
    
    # ADVANCED AUGMENTATION
    'mixup_alpha': 0.4,   # >0 enables MixUp
    'use_tta': True       # Test Time Augmentation
}

seed_everything(CONFIG['seed'])



In [ ]:

class CSIRODataset(Dataset):
    def __init__(self, df, img_dir, transform=None, mode='train'):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.mode = mode
        col_name = 'image_path' if 'image_path' in df.columns else df.columns[0]
        self.file_names = df[col_name].values
        if self.mode != 'test':
            self.labels = df[CONFIG['target_cols']].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_path = self.file_names[idx]
        full_path = get_image_path(file_path, self.img_dir)
        image = cv2.imread(full_path)
        if image is None: image = np.zeros((CONFIG['img_size'], CONFIG['img_size'], 3), dtype=np.uint8)
        else: image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform: image = self.transform(image=image)['image']
        if self.mode != 'test': return image, torch.tensor(self.labels[idx])
        return image

def get_image_path(filename, search_dir):
    p = os.path.join(search_dir, filename)
    if os.path.exists(p): return p
    p = os.path.join(search_dir, os.path.basename(filename))
    if os.path.exists(p): return p
    return filename

def get_transforms(img_size):
    MEAN = [0.485, 0.456, 0.406]
    STD = [0.229, 0.224, 0.225]
    return {
        'train': A.Compose([
            A.RandomResizedCrop(img_size, img_size, scale=(0.8, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.2), # Added for robustness
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ]),
        'valid': A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ]),
        # TTA Transforms (Flip + Scale)
        'tta_hflip': A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(img_size, img_size),
            A.Normalize(mean=MEAN, std=STD),
            ToTensorV2(),
        ])
    }



In [ ]:

class BiomassModel(nn.Module):
    def __init__(self, model_name, num_classes=5, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.n_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(self.n_features, 512),
            nn.BatchNorm1d(512),
            nn.SiLU(),
            nn.Dropout(0.3), # Increased dropout for regularization
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        return self.head(self.backbone(x))



In [ ]:

def hierarchical_reconciliation(preds):
    preds = np.maximum(preds, 0)
    green, dead, total = preds[:, 0], preds[:, 1], preds[:, 4]
    sum_comp = green + dead
    mask = sum_comp > 1e-6
    scale_factor = np.ones_like(total)
    scale_factor[mask] = total[mask] / sum_comp[mask]
    new_preds = preds.copy()
    new_preds[:, 0] = green * scale_factor
    new_preds[:, 1] = dead * scale_factor
    new_preds[:, 2] = np.minimum(preds[:, 2], new_preds[:, 0]) 
    return new_preds



In [ ]:

def run_inference():
    print("Starting INFERENCE...")
    weights = glob.glob("*.pth") + glob.glob("/kaggle/input/*/*.pth")
    if not weights: print("No weights found!"); return
    print(f"Found {len(weights)} models: {weights}")

    if not os.path.exists(CONFIG['test_csv']): return
    test_df = pd.read_csv(CONFIG['test_csv'])
    
    # 1. Base Dataset
    test_ds = CSIRODataset(test_df, CONFIG['img_dir'], transform=get_transforms(CONFIG['img_size'])['valid'], mode='test')
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    
    # 2. TTA Dataset (Horizontal Flip)
    if CONFIG['use_tta']:
        print("Test Time Augmentation (TTA) Enabled.")
        tta_ds = CSIRODataset(test_df, CONFIG['img_dir'], transform=get_transforms(CONFIG['img_size'])['tta_hflip'], mode='test')
        tta_loader = DataLoader(tta_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
    
    # 3. Load Models (Auto-detect backbone from filename if possible, else default)
    models = []
    for w in weights:
        # Determine architecture from filename or config
        if 'convnext' in w: arch = 'convnextv2_base.fcmae'
        else: arch = 'vit_base_patch14_dinov2.lvd142m' # Default
        
        m = BiomassModel(arch, pretrained=False)
        m.load_state_dict(torch.load(w, map_location=CONFIG['device']))
        m.to(CONFIG['device']).eval()
        models.append(m)
        
    final_preds = []
    with torch.no_grad():
        # Iterate Loaders (Base + TTA)
        loaders = [test_loader]
        if CONFIG['use_tta']: loaders.append(tta_loader)
        
        # We need to average across Loaders AND Models
        accumulated_preds = np.zeros((len(test_df), 5))
        
        for loader in loaders:
            loader_preds = []
            for images in tqdm(loader, desc="Infer"):
                images = images.to(CONFIG['device'])
                batch_preds = [m(images).cpu().numpy() for m in models]
                # Average models for this batch
                loader_preds.append(np.mean(batch_preds, axis=0))
            accumulated_preds += np.vstack(loader_preds)
            
        accumulated_preds /= len(loaders)
        final_preds = accumulated_preds
            
    all_preds = hierarchical_reconciliation(final_preds)
    submission = pd.DataFrame(all_preds, columns=CONFIG['target_cols'])
    if 'image_path' in test_df.columns: submission.insert(0, 'image_path', test_df['image_path'])
    submission.to_csv('submission.csv', index=False)
    print("Saved submission.csv")

if __name__ == "__main__":
    run_inference()

